In [ ]:
import csv
import pickle
import pandas as pd
import numpy as np

import quartz.grids as qg
from quartz.data import MoleculeData
from quartz.dataprocessing import assign_props, calculate_overlap
from quartz.grids import create_grid
from quartz.io import load_and_process

In [2]:
# mols = load_and_process(input_dir="tests/sample_data/", size=0.1)

In [3]:
# pickle.dump(mols, open("FX01.pkl", 'wb'))

In [4]:
with open("./tests/sample_data/FX01.pkl", "rb") as f:
    mols = pickle.load(f)

In [22]:
mols[0].res.keys()

dict_keys(['coordination number', 'dispersion coef', 'polarizibility', 'charges', 'fullpop', 'fod_s', 'fod_p', 'fod_d', 'fukui+', 'fukui-', 'fukui0'])

In [5]:
with open("tests/sample_data/FX_data.csv", "r", encoding="utf-8-sig") as f:
    csvreader = csv.DictReader(f)
    for row in csvreader:
        # print(row)
        for mol in mols:
            if mol.filename[0:4] is not None and mol.filename[0:4] == row["PDB ID "]:
                mol.pdb_code = row["PDB ID "]
                mol.ki = float(row["Ki (nM)"])
                mol.pki = float(row["pKi "])
                # print(f"Assigned PDB code {row['PDB ID ']} to {mol.filename}")
                break

mols.sort(key=lambda x: x.ki)

In [6]:
# from rdkit.Chem.Draw import IPythonConsole, MolsToGridImage

# MolsToGridImage([mol.rdmol for mol in mols], legends=[mol.pdb_code for mol in mols], subImgSize=(300, 200))


In [ ]:
for i in mols:
    assign_props(i, ["fukui-"])

In [8]:
# mols[3].res["fukui-"]

In [ ]:
grids = qg.create_grid(
    mols[0:5],
    props=["fukui-"],
    padding=5.0,
    size=0.5,
    threshold={"fukui-": 0.001},
    clamp_dist=0.5,
    clamp_falloff=1000,
)

Grid lengths:  35.0 33.0 42.0
Origin:  -2.72099029 61.69410415 92.5117578
Cell count:  70 66 84
Processing:  COc1cc(Cl)cc(C(=O)Nc2ccc(Cl)cn2)c1NC(=O)c1scc(CN(C)c2ncco2)c1Cl
Processing:  C[NH+](C)Cc1nccn1-c1ccc(NC(=O)c2cc(C(F)(F)F)nn2-c2ccc3onc(N)c3c2)c(F)c1
Processing:  COC(=O)[C@H](Cc1cccc(C(N)=[NH2+])c1)[C@@H](C)NC(=O)c1ccc(C2=CCN(O)C=C2)cc1
Processing:  COC(=O)[C@H](Cc1cccc(C(N)=[NH2+])c1)[C@@H](C)NC(=O)c1ccc(-c2cccc(C[NH3+])c2)cc1
Processing:  C[N@H+]1CC[N@H+](Cc2csc(C(=O)Nc3ccc(Cl)cc3C(=O)Nc3ccc(Cl)cc3)c2Cl)CC1


In [ ]:
calculate_overlap(grids[0:5], ["fukui-"])

{'polarizibility': np.float64(0.6522),
 'fukui+': np.float64(0.7899),
 'fukui-': np.float64(0.7619)}

In [ ]:
print(f"Grid origin: {grids[0].grid_props[1]}")
print(f"Grid shape: {[grids[0].npGrid[key].shape for key in grids[0].npGrid.keys()]}")
print(f"Grid spacing: {grids[0].grid_props[2]}")

Grid origin: [np.float64(66.69410415), np.float64(89.07602038)]
Grid shape: [(70, 66, 84), (70, 66, 84), (70, 66, 84), (70, 66, 84)]
Grid spacing: [np.float64(97.5117578), np.float64(128.57507585)]


In [12]:
qg.to_dxgrid(grids)

In [20]:
for key in grids[0].npGrid.keys():
    print(f"{key}:")
    print([max(x.res[key]) for x in mols[0:5]])
    print([np.max(grid.npGrid[key]) for grid in grids])

polarizibility:
[19.709, 8.972, 8.964, 8.883, 19.381]
[np.float16(1.261e+03), np.float16(574.0), np.float16(573.5), np.float16(568.5), np.float16(1.24e+03)]
fukui-:
[0.067, 0.094, 0.072, 0.1, 0.194]
[np.float16(4.29), np.float16(6.016), np.float16(2.31), np.float16(4.9), np.float16(12.414)]
fukui+:
[0.12, 0.056, 0.046, 0.054, 0.235]
[np.float16(6.18), np.float16(2.785), np.float16(2.438), np.float16(2.887), np.float16(14.04)]
fukui0:
[0.086, 0.062, 0.056, 0.061, 0.136]
[np.float16(4.03), np.float16(3.969), np.float16(1.858), np.float16(2.404), np.float16(8.09)]


In [14]:
df = pd.DataFrame([x.__dict__ for x in mols])

In [15]:
df["pki"] = pd.to_numeric(df["pki"])
df.sort_values("pki", inplace=True, ascending=False)